# Forensic LLM Graph

Run the full pipeline in one **T4 GPU** session. The first pass uses rules and spaCy on CPU. Minerva and Qwen use the GPU to score passages and review entities and relations. The final step builds the graph.

1. Enable the `HF_TOKEN` Colab Secret with read access to `sapienzanlp/Minerva-350M-base-v1.0`.
2. Upload the prepared `data/processed/documents.json` to `/content/` using the Files panel. Keep run number `05` for the 10,654-document corpus.
3. Choose **Runtime → Run all**, then download `/content/05_run.zip` from the Files panel.

If an execution stops while the runtime still has its files, rerun with the same number to reuse completed stages. Hybrid NER resumes from its checkpoint. An interrupted relation review restarts from its saved plan. Choose a new number for a separate run.


## 1. Set up Colab


In [ ]:
import hashlib
import json
import os
from pathlib import Path
import shutil
import subprocess
import sys
from zipfile import ZIP_DEFLATED, ZipFile

from google.colab import userdata

run_number = "05"  # @param {type:"string"}
if len(run_number) != 2 or not run_number.isascii() or not run_number.isdecimal():
    raise ValueError("Use two digits for the run number, for example 05.")

workspace = Path("/content")
uploaded_corpus = workspace / "documents.json"
if not uploaded_corpus.is_file():
    raise FileNotFoundError("Upload the prepared documents.json to /content/ first.")

if shutil.which("nvidia-smi") is None:
    raise RuntimeError("Select a T4 GPU runtime in Colab.")
subprocess.run(["nvidia-smi", "-L"], check=True)

try:
    hf_token = userdata.get("HF_TOKEN")
except (userdata.SecretNotFoundError, userdata.NotebookAccessError):
    raise RuntimeError("Add the HF_TOKEN Secret and enable access for this notebook.") from None
if not hf_token:
    raise RuntimeError("Set a Hugging Face read token in the HF_TOKEN Secret.")

repo = workspace / "forensic-llm-graph"
output_dir = repo / "outputs"
baseline_path = output_dir / f"{run_number}_ner_baseline.json"
ner_path = output_dir / f"{run_number}_ner_corpus.json"
plan_path = output_dir / f"{run_number}_relation_plan.json"
reviews_path = output_dir / f"{run_number}_relation_reviews.json"
graph_path = output_dir / f"{run_number}_relation_graph.json"

if not repo.exists():
    subprocess.run(
        ["git", "clone", "https://github.com/AntoFratta/forensic-llm-graph.git", str(repo)],
        check=True,
    )
elif not baseline_path.exists():
    subprocess.run(["git", "pull", "--ff-only"], cwd=repo, check=True)

revision = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=repo, text=True).strip()
print(f"Pipeline revision: {revision}", flush=True)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], cwd=repo, check=True,
)

from huggingface_hub import hf_hub_download

hybrid_config = json.loads((repo / "configs/hybrid.json").read_text(encoding="utf-8"))
minerva = hybrid_config["perplexity"]
hf_hub_download(minerva["model"], "config.json", revision=minerva["revision"], token=hf_token)
model_env = {**os.environ, "HF_TOKEN": hf_token}
del hf_token

def sha256(path):
    with path.open("rb") as handle:
        return hashlib.file_digest(handle, "sha256").hexdigest()


def run_stage(path, stage, *args, environment=None, **expected):
    if path.is_file():
        print(f"Reusing {path.name}.")
    else:
        command = [sys.executable, "-u", *map(str, args), "--output", str(path)]
        with subprocess.Popen(
            command, cwd=repo, env=environment, stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT, text=True, encoding="utf-8",
        ) as process:
            try:
                for line in process.stdout:
                    if line.startswith("Perplexity:"):
                        done, total = map(int, line.split(":", 1)[1].split("/"))
                        if total > 100 and done % 100 and done != total:
                            continue
                    print(line, end="", flush=True)
                return_code = process.wait()
            except KeyboardInterrupt:
                process.terminate()
                process.wait()
                raise
        if return_code:
            raise subprocess.CalledProcessError(return_code, command)

    with path.open(encoding="utf-8") as handle:
        metadata = json.load(handle)["run"]
    if metadata.get("stage") != stage or any(metadata.get(key) != value for key, value in expected.items()):
        raise ValueError(f"{path.name} does not match this run. Use a new run number.")
    return metadata


## 2. Extract entities with rules and spaCy

The script processes the full corpus on CPU and saves document text and candidate spans.


In [ ]:
corpus_path = repo / "data/processed/documents.json"
corpus_path.parent.mkdir(parents=True, exist_ok=True)
shutil.copyfile(uploaded_corpus, corpus_path)

baseline = run_stage(
    baseline_path, "first_pass", "scripts/run_pipeline.py",
    input_sha256=sha256(corpus_path), mode="rules_and_spacy",
    git={"commit": revision, "uncommitted_changes": False},
)
document_count = baseline["document_count"]
print(f"Documents: {document_count:,}. First-pass candidates: {baseline['candidate_count']:,}.")


## 3. Score passages and review entities

Minerva scores all passages and Qwen reviews the selected passages under the configured budget. The script saves checkpoints as it runs.


In [ ]:
checkpoint = ner_path.with_suffix(".checkpoints.jsonl")
resume = ["--resume"] if checkpoint.is_file() else []
ner = run_stage(
    ner_path, "hybrid", "scripts/run_hybrid.py", "--input", baseline_path, *resume,
    environment=model_env, input_sha256=sha256(baseline_path), scope="extended",
    document_count=document_count, git=baseline["git"],
)
print(f"Entity review statuses: {ner['review_status_counts']}.")


## 4. Review explicit relations and build the graph

The plan selects up to 100 passages for Qwen. The builder validates the review against the NER input and stores evidence offsets for the graph edges.


In [ ]:
ner_sha256 = sha256(ner_path)
plan = run_stage(
    plan_path, "relation_review_plan", "scripts/review_relation_graph.py", "prepare",
    "--input", ner_path, input_sha256=ner_sha256, scope="corpus",
)
reviews = run_stage(
    reviews_path, "relation_review", "scripts/review_relation_graph.py", "run",
    "--plan", plan_path, environment=model_env, input_sha256=ner_sha256,
    plan_sha256=sha256(plan_path), selected_passages=plan["selected_passages"],
)
print(f"Relation review statuses: {reviews['statuses']}.")

graph = run_stage(
    graph_path, "relation_graph", "scripts/build_relation_graph.py",
    "--input", ner_path, "--relation-reviews", reviews_path,
    input_sha256=ner_sha256, relation_review_sha256=sha256(reviews_path),
)
if graph["nodes_by_type"].get("DOCUMENT", 0) != document_count:
    raise ValueError("The graph does not cover all documents in the input.")
print(f"Nodes: {graph['nodes_by_type']}")
print(f"Edges: {graph['edges_by_type']}")


## 5. Download the run

Download the ZIP from the Colab Files panel and extract its five JSON files into the repository's local `outputs/` folder.


In [ ]:
bundle_path = workspace / f"{run_number}_run.zip"
temporary_bundle = bundle_path.with_suffix(".zip.tmp")
with ZipFile(temporary_bundle, "w", compression=ZIP_DEFLATED, compresslevel=1) as bundle:
    for path in (baseline_path, ner_path, plan_path, reviews_path, graph_path):
        bundle.write(path, arcname=path.name)
temporary_bundle.replace(bundle_path)
print(f"Download {bundle_path} from the Colab Files panel.")
